<a href="https://colab.research.google.com/github/jagtappranav2721-cpu/ML-Internship-FlyRank/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

## Lane Chosen:
**Refresh / Content Opportunity Scoring**

## 1. My Lane and Why

I selected the **Refresh / Content Opportunity Scoring** lane.

Large websites often contain thousands of pages, making manual review inefficient.
This lane focuses on identifying which pages should be reviewed first using observable search and engagement signals.

Rather than predicting rankings, the goal is to prioritize limited editorial effort toward pages that show evidence of decline or missed opportunity.

I chose this lane because it combines practical business value with explainable machine learning and produces recommendations that humans can inspect before taking action.

## 2. Research Question

### Research Question

Which content pages should be reviewed first for refresh, expansion, monitoring, or protection based on observable search performance signals?

### Decision

Help a content team decide which pages deserve review before others.

### Unit of Analysis

Each row represents one content page from the starter dataset.

### Output

A ranked list of pages with a suggested review priority, confidence score, and explanation.

### Action

Editors can:

- Refresh content
- Expand information
- Improve CTR
- Monitor performance
- Leave the page unchanged

### Cost of a Wrong Recommendation

- False Positive:
A page is reviewed unnecessarily, wasting editor time.

- False Negative:
An important declining page is ignored, resulting in lost traffic and missed opportunities.

### Why Machine Learning Helps

- Manual review does not scale to thousands of pages.
- Machine learning can identify patterns in historical search and engagement data to rank pages by review priority while keeping humans responsible for the final decision.

## 3. Quick look at the data (2-3 real numbers)




In [21]:
!git clone https://github.com/jagtappranav2721-cpu/ML-Internship-FlyRank.git

fatal: destination path 'ML-Internship-FlyRank' already exists and is not an empty directory.


In [22]:
!find . -name "*.csv"

./ML-Internship-FlyRank/outputs/refresh_queue_sample.csv
./ML-Internship-FlyRank/data/raw/content_refresh_anonymized.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_train.csv
./sample_data/california_housing_test.csv
./sample_data/mnist_test.csv


In [23]:
import pandas as pd

df = pd.read_csv("./ML-Internship-FlyRank/outputs/refresh_queue_sample.csv")
df.head()

,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   final_rank              200 non-null    int64  
 1   content_id              200 non-null    object 
 2   client_id               200 non-null    object 
 3   final_refresh_score     200 non-null    float64
 4   best_model_name         200 non-null    object 
 5   best_model_probability  200 non-null    float64
 6   baseline_refresh_score  200 non-null    float64
 7   confidence              200 non-null    object 
 8   suggested_action        200 non-null    object 
 9   final_reason_codes      200 non-null    object 
 10  is_declining_label      200 non-null    int64  
 11  impressions_90d         200 non-null    int64  
 12  clicks_90d              200 non-null    int64  
 13  sessions_90d            200 non-null    int64  
 14  avg_position            200 non-null    fl

In [25]:
df.describe()

,final_rank,final_refresh_score,best_model_probability,baseline_refresh_score,is_declining_label,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
mean,100.500000,77.948590,0.801775,0.686863,0.985000,10855.935000,21.435000,36.940000,14.141000,0.157100,163.655000,77.220000,2570.055000
std,57.879185,1.051172,0.035123,0.077209,0.121857,10774.862297,32.528083,33.969487,9.751816,0.166452,48.537671,40.308753,1477.986227
min,1.000000,76.762175,0.691605,0.559155,0.000000,625.000000,0.000000,2.000000,1.100000,0.000000,106.000000,20.000000,0.000000
25%,50.750000,77.219425,0.784493,0.630901,1.000000,2515.250000,2.000000,11.750000,5.475000,0.050000,139.000000,20.000000,1407.250000
50%,100.500000,77.574389,0.808271,0.671554,1.000000,4783.000000,5.000000,25.000000,11.650000,0.110000,144.000000,104.000000,1587.000000
75%,150.250000,78.461687,0.829399,0.726889,1.000000,17800.500000,25.750000,51.500000,21.450000,0.210000,165.000000,104.000000,3969.750000
max,200.000000,81.636697,0.853060,0.908853,1.000000,45127.000000,161.000000,161.000000,39.400000,0.940000,333.000000,194.000000,6962.000000


In [26]:
df.columns

Index(['final_rank', 'content_id', 'client_id', 'final_refresh_score',
       'best_model_name', 'best_model_probability', 'baseline_refresh_score',
       'confidence', 'suggested_action', 'final_reason_codes',
       'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d',
       'avg_position', 'ctr', 'content_age_days', 'days_since_last_update',
       'word_count', 'trend_direction', 'competition_level', 'content_type',
       'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier',
       'impression_tier', 'position_tier'],
      dtype='object')

In [27]:
df["suggested_action"].value_counts()

,count
suggested_action,
refresh_and_review_ctr,130
refresh,35
refresh_and_review_engagement,35


In [28]:
df["trend_direction"].value_counts()

,count
trend_direction,
down,197
stable,2
up,1


Observations from the above two cells:

A substantial number of pages require a refresh or review, demonstrating that prioritization is a realistic problem. Since editorial resources are limited, ranking pages by refresh priority can help content teams focus on the highest-impact opportunities first.

In [29]:
print("Rows:", len(df))

print("Unique pages:", df["content_id"].nunique())

print("Average impressions:", df["impressions_90d"].mean())

print("Average CTR:", df["ctr"].mean())

Rows: 200
Unique pages: 200
Average impressions: 10855.935
Average CTR: 0.15710000000000002


**Quick Look at the Data:**

**Observations**:

1.The dataset contains 200 unique content pages, meaning each row represents a different page rather than duplicate observations. This makes the dataset suitable for page-level prioritization.

2.The average page received approximately 10,856 impressions over the last 90 days, indicating that the pages have measurable search visibility. Since these pages are already attracting impressions, improving underperforming pages could have a meaningful impact on user engagement.

3.The average CTR is approximately 15.7%, suggesting that user engagement varies across pages. Pages with relatively low CTR despite good visibility may represent opportunities for content refreshes or metadata improvements.

4.The dataset also contains features such as content age, trend direction, average position, competition level, word count, and suggested actions.

These variables are directly relevant to prioritizing which pages should be reviewed first, making the **Refresh / Content Opportunity Scoring**  lane an appropriate choice for this project.

# 4. Careful Words: What I Can and Can't Claim

A good machine learning project should support decision-making without making claims that the data cannot justify. The goal of this project is to identify pages that are good candidates for content refresh, not to guarantee improvements in search performance.

## What I Can Claim

- This project can identify patterns in observable search and content performance signals, such as impressions, click-through rate (CTR), content age, average position, and trend direction.
- It can rank content pages by their likelihood of benefiting from a refresh, helping content teams prioritize their review efforts.
- The recommendations generated by the model are intended to support human decision-making rather than replace it.

## What I Cannot Claim

- I cannot claim that refreshing a page will definitely improve its search rankings, traffic, or user engagement.
- I cannot determine the exact factors or algorithms used by search engines for ranking content.
- The relationships observed in this dataset are correlational and should not be interpreted as proof of causation.
- The model's recommendations should always be reviewed by content editors before any changes are implemented.

## Limitations

- This analysis is based only on the provided starter dataset, which contains 200 content pages and may not represent every type of website or content.
- The dataset contains historical observations and does not include experimental evidence or future outcomes.
- Additional data, domain expertise, and continuous validation would improve the reliability of the recommendations in a real-world production environment.

# 5. Self Check

After completing this notebook, I believe I have addressed the key requirements of this week's task.

- I selected **Lane 2 – Refresh / Content Opportunity Scoring** and explained why I chose it.
- I clearly defined my research question, the decision it supports, the action someone can take, and the cost of making an incorrect recommendation.
- I explored the starter dataset and used real statistics, including the number of content pages, average impressions, and average CTR, to justify why this problem is worth studying.
- I explained both the strengths and limitations of my proposed approach and avoided making claims that cannot be supported by the available data.
- Most importantly, I focused on understanding the business problem first instead of jumping directly into building a machine learning model.

Overall, I am confident that my research question is well-defined and provides a strong foundation for the remaining weeks of the project.